In [1]:
import pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
from common import build_windows, CorrosionSequenceDataset, CorrosionLSTM, ALL_FEATURES

EA, R_CONST = 45_000.0, 8.314
LAMBDA_PHYSICS = 0.1

def external_physics_loss(pred_external, temperature_c_last_step):
    """Only the EXTERNAL head is constrained by Arrhenius — MIC/internal
    corrosion has no equivalent closed-form law, so the internal head
    stays purely data-driven (an explicit, documented scoping choice)."""
    temperature_k = temperature_c_last_step + 273.15
    expected = torch.exp(-EA / (R_CONST * temperature_k))
    expected = expected / expected.mean()
    predicted = pred_external / (pred_external.mean() + 1e-8)
    return torch.mean((predicted - expected) ** 2)

df = pd.read_csv("../data/simulated_corrosion.csv")
means, stds = df[ALL_FEATURES].mean(), df[ALL_FEATURES].std()
df[ALL_FEATURES] = (df[ALL_FEATURES] - means) / stds

X, y = build_windows(df)
temp_idx = ALL_FEATURES.index("temperature_C")

loader = DataLoader(CorrosionSequenceDataset(X, y), batch_size=64, shuffle=True)
model = CorrosionLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
mse = nn.MSELoss()

for epoch in range(20):
    total = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        pred = model(xb)  # [:, 0] = external, [:, 1] = internal
        temp_last_step = xb[:, -1, temp_idx] * stds["temperature_C"] + means["temperature_C"]
        loss = (mse(pred, yb)
                + LAMBDA_PHYSICS * external_physics_loss(pred[:, 0], temp_last_step))
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1}  loss={total/len(loader.dataset):.4f}")

torch.save(model.state_dict(), "pinn_lstm_model.pth")
print("Saved pinn_lstm_model.pth")

epoch 1  loss=5.1306
epoch 2  loss=3.4810
epoch 3  loss=3.4582
epoch 4  loss=3.4067
epoch 5  loss=3.3679
epoch 6  loss=3.3165
epoch 7  loss=3.2375
epoch 8  loss=3.1281
epoch 9  loss=3.0178
epoch 10  loss=2.7981
epoch 11  loss=2.5911
epoch 12  loss=2.3086
epoch 13  loss=2.0558
epoch 14  loss=1.7589
epoch 15  loss=1.5348
epoch 16  loss=1.3352
epoch 17  loss=1.1674
epoch 18  loss=1.0271
epoch 19  loss=0.8930
epoch 20  loss=0.7850
Saved pinn_lstm_model.pth
